# **Inference Pipeline**

## **Refactorización para Producción (Productionizing).**

### **¿Como lo hicimos? (La Anatomía del Script)**

Si miras el script con rayos X, verás que está construido en 3 bloques lógicos que "roban" código de tus notebooks anteriores:

## **Bloque A: La Entrada (El Atajo)**
- **Código:** df = pd.read_parquet(MASTER_FILE)

- **El Truco:** En lugar de repetir todo el trabajo sucio del Notebook 02 (cargar Excel, limpiar headers, arreglar fechas, unir con clima...), el script asume que ese trabajo ya se hizo. Carga directamente el archivo .parquet limpio.

- **Por qué:** Eficiencia pura. No reinventamos la rueda cada vez que queremos predecir.

## **Bloque B: La Memoria (Feature Engineering)**
- **Código: La función engineer_features(df_input).**

- **El Truco:** Copié y pegué la lógica exacta del Notebook 03 (los lags, las ventanas móviles, la estacionalidad) y la envolví en una función.

- **Por qué:** Esta es la regla de oro del Machine Learning: "El modelo debe recibir los datos en producción EXACTAMENTE igual que como los recibió en el entrenamiento". Si entrenaste con casos_lag_4, no puedes predecir sin calcular primero casos_lag_4.

## **Bloque C:** El Bucle del Tiempo (Predicción Recursiva)
- **Código:** El ciclo for date in future_dates:

- **El Truco:** Esto es algo nuevo que no estaba en los notebooks.

- El modelo predice la Semana 1.

- El script toma esa predicción y la escribe en la base de datos temporal como si fuera un dato real.

- Para predecir la Semana 2, el modelo mira hacia atrás y "ve" la predicción de la Semana 1 (que acabamos de inventar) y la usa para calcular su lag.

**Por qué:** Porque sin este truco, no podríamos predecir más allá de 1 semana en el futuro. Necesitamos "alucinar" el pasado inmediato para predecir el futuro lejano.

### **3. ¿Por que lo hicimos así? (El Objetivo Tableau)**

Aquí está la razón estratégica de por qué evitamos otras partes y por qué el output es un CSV simple.

Tableau es "Tonto" (pero bonito): Tableau no sabe ejecutar Python. No sabe qué es un XGBoost. No puede calcular lags. Solo sabe leer tablas y hacer gráficos.

El Script es el "Traductor": El script hace todo el trabajo matemático pesado y difícil (la IA) y le entrega a Tableau un archivo que hasta un niño podría entender:

Fecha: 2025-01-01

Casos: 150

Tipo: Pronóstico

Re-entrenamiento Total: Notarás que en el script no dividí en Train/Test. Entrené con TODO el historial.

Por qué: En el notebook dividimos para evaluar si el modelo era bueno. Pero ahora que sabemos que es bueno, queremos que aprenda de todos los datos posibles, incluso los de la semana pasada, para que su predicción de mañana sea la mejor posible. No nos guardamos nada.

